A. Studi Kasus: Klasifikasi Otomatis Pemicu Deforestasi Hutan Hujan Tropis Berbasis Indeks Biofisik Spektral Satelit
Latar Belakang & Masalah: Sistem peringatan deforestasi satelit global (seperti GLAD / RADD Alerts) hanya mendeteksi bahwa kanopi pohon telah hilang di koordinat tertentu, namun tidak mampu menjelaskan faktor pemicunya. Kementerian Lingkungan Hidup dan auditor rantai pasok butuh mengetahui secara otomatis apakah hilangnya pohon disebabkan oleh ekspansi perkebunan kelapa sawit monokultur, pembukaan jalan logging, tambang ilegal terbuka, atau kebakaran hutan.
Solusi:
Mengekstraksi pita spektral satelit optik: Green (
𝐵
3
B
3
	​

), Red (
𝐵
4
B
4
	​

), Near-Infrared / NIR (
𝐵
8
B
8
	​

), dan Short-Wave Infrared / SWIR (
𝐵
11
B
11
	​

).
Menghitung indeks matematis biofisik vegetasi:
NDVI
=
𝑁
𝐼
𝑅
−
𝑅
𝑒
𝑑
𝑁
𝐼
𝑅
+
𝑅
𝑒
𝑑
,
NBR
=
𝑁
𝐼
𝑅
−
𝑆
𝑊
𝐼
𝑅
𝑁
𝐼
𝑅
+
𝑆
𝑊
𝐼
𝑅
,
NDWI
=
𝐺
𝑟
𝑒
𝑒
𝑛
−
𝑁
𝐼
𝑅
𝐺
𝑟
𝑒
𝑒
𝑛
+
𝑁
𝐼
𝑅
NDVI=
NIR+Red
NIR−Red
	​

,NBR=
NIR+SWIR
NIR−SWIR
	​

,NDWI=
Green+NIR
Green−NIR
	​

Melatih model Random Forest Classifier untuk memprediksi 4 kategori pendorong deforestasi berdasarkan profil tanda tangan spektral (spectral signatures).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

def studi_kasus_6_satellite_ai_deforestation():
    np.random.seed(42)
    n_per_class = 250
    kategori = ['Kelapa Sawit Industri', 'Pertambangan Terbuka', 'Kebakaran Hutan', 'Jalan Logging / Akses']

    # 1. Simulasi Reflektansi Spektral Sentinel-2 untuk Setiap Pemicu:
    # Sawit: Re-vegetasi seragam, NIR sedang, NDVI moderat ~0.45
    nir_sawit = np.random.normal(0.42, 0.04, n_per_class)
    red_sawit = np.random.normal(0.12, 0.02, n_per_class)
    swir_sawit = np.random.normal(0.18, 0.03, n_per_class)
    green_sawit = np.random.normal(0.10, 0.02, n_per_class)

    # Tambang: Tanah terbuka & bebatuan gundul, Red tinggi, SWIR tinggi, NDVI rendah
    nir_tambang = np.random.normal(0.20, 0.03, n_per_class)
    red_tambang = np.random.normal(0.24, 0.03, n_per_class)
    swir_tambang = np.random.normal(0.32, 0.04, n_per_class)
    green_tambang = np.random.normal(0.14, 0.02, n_per_class)

    # Kebakaran: Abu/arang hitam, NIR anjlok drastis, NBR bernilai sangat negatif
    nir_bakar = np.random.normal(0.12, 0.02, n_per_class)
    red_bakar = np.random.normal(0.18, 0.02, n_per_class)
    swir_bakar = np.random.normal(0.38, 0.04, n_per_class)
    green_bakar = np.random.normal(0.08, 0.02, n_per_class)

    # Jalan Logging: Pola tanah padat memanjang linier
    nir_jalan = np.random.normal(0.25, 0.03, n_per_class)
    red_jalan = np.random.normal(0.19, 0.02, n_per_class)
    swir_jalan = np.random.normal(0.25, 0.03, n_per_class)
    green_jalan = np.random.normal(0.11, 0.02, n_per_class)

    nir = np.concatenate([nir_sawit, nir_tambang, nir_bakar, nir_jalan])
    red = np.concatenate([red_sawit, red_tambang, red_bakar, red_jalan])
    swir = np.concatenate([swir_sawit, swir_tambang, swir_bakar, swir_jalan])
    green = np.concatenate([green_sawit, green_tambang, green_bakar, green_jalan])
    y = np.array([0]*n_per_class + [1]*n_per_class + [2]*n_per_class + [3]*n_per_class)

    # 2. Formula Biofisik Citra Satelit (Remote Sensing Indices)
    ndvi = (nir - red) / (nir + red + 1e-6)
    nbr = (nir - swir) / (nir + swir + 1e-6)
    ndwi = (green - nir) / (green + nir + 1e-6)

    df_satelit = pd.DataFrame({
        'Band_Green': green, 'Band_Red': red, 'Band_NIR': nir, 'Band_SWIR': swir,
        'NDVI': ndvi, 'NBR': nbr, 'NDWI': ndwi, 'Driver_Class': y
    })

    X = df_satelit.drop(columns=['Driver_Class'])
    y = df_satelit['Driver_Class']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

    # 3. Klasifikasi Pemicu Deforestasi dengan AI
    rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_test)

    print("=== KLASIFIKASI AI UNTUK ATRIBUSI PEMICU DEFORESTASI ===")
    print("Laporan Performa Model per Kategori Pendorong:")
    print(classification_report(y_test, y_pred, target_names=kategori))
    print("Kontribusi Fitur Spektral Teratas (Feature Importance):")
    imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
    for feat, val in imp.items():
        print(f"  - {feat:11s}: {val*100:.2f}%")

if __name__ == '__main__':
    studi_kasus_6_satellite_ai_deforestation()